In [ ]:
pip install -U transformers peft bitsandbytes accelerate datasets


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 72.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 18.7 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
import pandas as pd
import re
import string
import nltk
from nltk.corpus import stopwords

nltk.download("stopwords")

# ================= CONFIG =================
INPUT_CSV = "/content/train_en_2000.csv"
OUTPUT_CSV = "train_clean.csv"

LOWERCASE = True
REMOVE_STOPWORDS = True

# Causal words we must NOT remove
CAUSAL_WHITELIST = {
    "because", "due", "due to", "caused", "caused by",
    "resulted", "resulted from", "driven", "driven by",
    "as", "since"
}

STOPWORDS = set(stopwords.words("english"))

# Remove causal words from stopwords
STOPWORDS = {w for w in STOPWORDS if w not in CAUSAL_WHITELIST}

def clean_text(text, aggressive=False):
    if pd.isna(text):
        return ""
    return str(text)

# ================= MAIN =================
def main():
    df = pd.read_csv(
    INPUT_CSV,
    engine="python",      # more tolerant than C-engine
    sep=";",
    quotechar='"',
    escapechar="\\",
    on_bad_lines="skip"   # skip corrupted rows instead of crashing
)

    print("Cleaning dataset...")

    df["context"] = df["context"].apply(lambda x: clean_text(x, aggressive=True))
    df["question"] = df["question"].apply(lambda x: clean_text(x, aggressive=True))
    df["answer"] = df["answer"].apply(lambda x: clean_text(x, aggressive=False))

    df.to_csv(OUTPUT_CSV, index=False)

    print(f"Cleaned dataset saved to: {OUTPUT_CSV}")
    print("Sample cleaned row:")
    print(df.iloc[0])

if __name__ == "__main__":
    main()


Cleaning dataset...
Cleaned dataset saved to: train_clean.csv
Sample cleaned row:
id                                                          1
context     – To provide a significant proportion of perfo...
question    What is the consequence of providing a signifi...
answer      aligning management with shareholders' interes...
Name: 0, dtype: object


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model

BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
TRAIN_FILE = "/content/train_clean.csv"
OUTPUT_DIR = "./qlora_fincausal"

In [ ]:
# ================= TOKENIZER =================
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

In [ ]:
# ================= QLoRA: 4-BIT BASE MODEL =================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto"
)

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

In [ ]:
lora_config = LoraConfig(
    r=32,                     # better capacity than 16
    lora_alpha=64,            # usually 2 * r
    lora_dropout=0.05,        # good regularization
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"   # improves learning a lot
    ]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 66,060,288 || all params: 4,088,528,384 || trainable%: 1.6157


In [ ]:
# ================= DATA =================
dataset = load_dataset("csv", data_files=TRAIN_FILE)["train"]

def format_example(ex):
    return {
        "text": f"""Context:
{ex['context']}

Question:
{ex['question']}

Answer:
{ex['answer']}"""
    }

dataset = dataset.map(format_example)

def tokenize(ex):
    return tokenizer(
        ex["text"],
        truncation=True,
        max_length=512,
        padding=False
    )

dataset = dataset.map(tokenize, remove_columns=dataset.column_names)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    per_device_train_batch_size=2,
    gradient_accumulation_steps=16,   # effective batch = 32

    learning_rate=1e-4,               # safer than 2e-4
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,

    num_train_epochs=3,               # increase if small dataset
    max_grad_norm=1.0,

    fp16=True,                        # use bf16 if A100/H100
    logging_steps=25,

    save_strategy="epoch",
    save_total_limit=2,

    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

trainer.train()
model.save_pretrained(OUTPUT_DIR)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss
25,1.949648
50,1.658691


OutOfMemoryError: CUDA out of memory. Tried to allocate 552.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 37.81 MiB is free. Including non-PyTorch memory, this process has 14.52 GiB memory in use. Of the allocated memory 13.97 GiB is allocated by PyTorch, and 424.92 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

class HiddenStateExtractor:
    def __init__(self, base_model, qlora_path, device="cuda"):
        self.device = device

        self.tokenizer = AutoTokenizer.from_pretrained(base_model, use_fast=True)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16
        )

        base = AutoModelForCausalLM.from_pretrained(
            base_model,
            quantization_config=bnb_config,
            device_map="auto"
        )

        self.model = PeftModel.from_pretrained(base, qlora_path)
        self.model.eval()


In [ ]:
class FinCausalGenerator:
    def __init__(self, base_model, qlora_path, device="cuda"):
        self.device = device
        self.extractor = HiddenStateExtractor(base_model, qlora_path, device)
        self.tokenizer = self.extractor.tokenizer
        self.model = self.extractor.model

    def generate(self, context, question):
        prompt = f"""
You are a financial causal extraction system.

Task:
Extract the correct causal answer from the context using the question.

Instructions:
- The answer is explicitly stated in the context.
- Use ONLY information from the context.
- Identify the correct cause-and-effect relationship.
- Produce ONE short, complete sentence.
- Prefer copying exact phrases from the context.
- Do NOT explain, summarize, or add information.
- Do NOT restate the question.
- Do NOT use background knowledge.

Context:
{context}

Question:
{question}

Answer:
""".strip()

        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True
        ).to(self.device)

        with torch.no_grad():
            output = self.model.generate(
                **inputs,
                max_new_tokens=40,
                do_sample=False,
                temperature=0.0,
                repetition_penalty=1.1,
                use_cache=False,        # 🔥 important for Colab
                pad_token_id=self.tokenizer.eos_token_id,
                eos_token_id=self.tokenizer.eos_token_id
            )

        answer = self.tokenizer.decode(
            output[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True
        )

        # hard cleanup
        return answer.split("\n")[0].strip().strip('"')


In [ ]:
import pandas as pd
import csv
from tqdm import tqdm

BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
QLORA_PATH = "./qlora_fincausal"

generator = FinCausalGenerator(BASE_MODEL, QLORA_PATH)

df = pd.read_csv("/content/test_en_500_clean - test_en_500_clean.csv.csv")

tqdm.pandas(desc="Generating answers")

df["answer"] = df.progress_apply(
    lambda r: generator.generate(r["context"], r["question"]),
    axis=1
)

df.to_csv(
    "final_submission.csv",
    sep=";",
    index=False,
    header=False,
    quoting=csv.QUOTE_ALL,
    encoding="utf-8"
)

print("✅ final_submission.csv generated successfully")


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
